In [1]:
from numpy import where
import numpy as np
import seaborn as sns

from PIL import Image
from PIL import ImageEnhance
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from keras.utils import to_categorical

2025-12-10 15:47:33.898648: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-10 15:47:35.845401: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-10 15:47:45.039102: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, LSTM, 
                                      Bidirectional, Dense, Dropout, 
                                      BatchNormalization, Concatenate)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr

import tensorflow as tf
from sklearn.model_selection import KFold
from itertools import product


In [3]:
import os
path = "/scratch/prj/dn_hamidlab_msc_proj/project_k20029203/outputs/nmd"

In [4]:
import os
os.environ["RPY2_CFFI_MODE"] = "ABI" 
os.environ['R_HOME'] = '/software/spackages_v0_21_prod/apps/linux-ubuntu22.04-zen2/gcc-13.2.0/r-4.5.1-fcdalrrhty67okjue6qglrbsjqnswlnx/rlib/R'

In [5]:
from rpy2.robjects import r
import numpy as np

one_hot_list = r.readRDS("/scratch/prj/dn_hamidlab_msc_proj/project_k20029203/outputs/nmd/one_hot_list.rds")
one_hot_1_list = r.readRDS("/scratch/prj/dn_hamidlab_msc_proj/project_k20029203/outputs/nmd/one_hot_1_list.rds")

print("Length:", len(one_hot_list))

Length: 257602


In [6]:
from rpy2.robjects.conversion import Converter
from rpy2.robjects import numpy2ri
import numpy as np

converter = Converter('numpy converter')
converter += numpy2ri.converter

with converter.context():
    onehot_np = np.asarray(one_hot_list)

with converter.context():
    onehot_1_np = np.asarray(one_hot_1_list)

In [7]:
print(onehot_np.shape)

(257602, 103, 4)


In [8]:
x_np = onehot_np.reshape(onehot_np.shape[0], -1)
print(x_np.shape)

(257602, 412)


In [9]:
log_fc = r.readRDS("/scratch/prj/dn_hamidlab_msc_proj/project_k20029203/outputs/nmd/for_seq.rds")
log_fc_1 = r.readRDS("/scratch/prj/dn_hamidlab_msc_proj/project_k20029203/outputs/nmd/for_seq_1.rds")

with converter.context():
    log_fc_np = np.asarray(log_fc)

with converter.context():
    log_fc_1_np = np.asarray(log_fc_1)

print(log_fc_np.shape)

(257602,)


In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(
    onehot_np, log_fc_np, test_size=0.20, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

In [ ]:
def model(lstm_units=128, dropout_rate1=0.3, dropout_rate2=0.2, 
                 dense_units=64, learning_rate=1e-3):

    return model

In [19]:
seq_input = Input(shape=(103, 4), name='sequence_input')
    
x = LSTM(128, return_sequences=False, 
             kernel_regularizer=tf.keras.regularizers.l2(0.01))(seq_input)
x = Dropout(0.3)(x)
    
x = Dense(64, activation="relu",
              kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
x = Dropout(0.2)(x)
    
logfc_output = Dense(1, activation="linear", name='logfc_output')(x)
    
model = Model(inputs=seq_input, outputs=logfc_output)
    
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',  
    metrics=['mae']  
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)     │ (None, 103, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │        68,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logfc_output (Dense)            │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 76,417 (298.50 KB)

 Trainable params: 76,417 (298.50 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
def model(lstm_units, dropout_rate1, dropout_rate2, dense_units, learning_rate, input_shape):
    model = Sequential([
        LSTM(lstm_units, input_shape=input_shape, return_sequences=False),
        Dropout(dropout_rate1),
        Dense(dense_units, activation='relu'),
        Dropout(dropout_rate2),
        Dense(1)
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    
    return model


m = model(
    lstm_units=128,
    dropout_rate1=0.3,
    dropout_rate2=0.2,
    dense_units=64,
    learning_rate=1e-3,
    input_shape=(X_train.shape[1], X_train.shape[2])
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
]

history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks
)

test_loss, test_mae = m.evaluate(X_test, y_test)

NameError: name 'Sequential' is not defined

In [11]:
print(f"Mean: {np.mean(y_train):.4f}")
print(f"Std: {np.std(y_train):.4f}")
print(f"Min: {np.min(y_train):.4f}")
print(f"Max: {np.max(y_train):.4f}")
print(f"Median: {np.median(y_train):.4f}")
print(f"IQR: {np.percentile(y_train, 75) - np.percentile(y_train, 25):.4f}")

Mean: -0.7202
Std: 5.7318
Min: -31.0759
Max: 36.1173
Median: -0.1907
IQR: 7.1489


In [16]:
#lstm units only with batch norm + bidirectional
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import BatchNormalization, Bidirectional

def model(lstm_units, dropout_rate1, dropout_rate2, dense_units, learning_rate, input_shape):
    model = Sequential([
        Bidirectional(
            LSTM(
                lstm_units, 
                input_shape=input_shape, 
                return_sequences=False,
                kernel_regularizer=l2(0.001),
                recurrent_regularizer=l2(0.001)
            )
        ),
        BatchNormalization(),
        Dropout(dropout_rate1),
        Dense(
            dense_units, 
            activation='relu',
            kernel_regularizer=l2(0.001)
        ),
        BatchNormalization(),
        Dropout(dropout_rate2),
        Dense(1, kernel_regularizer=l2(0.001))
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss='huber',
        metrics=['mae', 'mse']
    )
    
    return model


m = model(
    lstm_units=128,
    dropout_rate1=0.3,
    dropout_rate2=0.2,
    dense_units=64,
    learning_rate=1e-4,
    input_shape=(X_train.shape[1], X_train.shape[2])
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, min_delta=0.001),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, min_delta=0.001)
]

history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=64,
    callbacks=callbacks,
    class_weight=None
)

ImportError: cannot import name 'Sequential' from 'tensorflow.keras.layers' (/cephfs/volumes/hpc_home/k20029203/fca66952-06d6-4b07-b95c-4b56cba53523/jvenv/lib/python3.11/site-packages/keras/_tf_keras/keras/layers/__init__.py)

In [ ]:
test_loss, test_mae, test_mse = m.evaluate(X_test, y_test)

predictions = m.predict(X_test)
from scipy.stats import pearsonr
correlation, _ = pearsonr(y_test.flatten(), predictions.flatten())

In [19]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import BatchNormalization, Bidirectional

def model(input_shape):
    input = input_shape
    
    x= Bidirectional(LSTM(256, 
             return_sequences=False,
             kernel_regularizer=l2(0.001),
             recurrent_regularizer=l2(0.001)))(input)
            
    x = BatchNormalization()(x),
    x = Dropout(0.2)(x),

    x= Bidirectional(LSTM(128, 
             return_sequences=False,
             kernel_regularizer=l2(0.001),
             recurrent_regularizer=l2(0.001)))(x)

    x = BatchNormalization()(x),
    x = Dropout(0.2)(x),
    
    model.compile(
        optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
        loss='huber',
        metrics=['mae', 'mse']
    )
    
    return model


m = model(
    input_shape=(X_train.shape[1], X_train.shape[2])
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, min_delta=0.001),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, min_delta=0.001)
]

history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=128,
    callbacks=callbacks,
    class_weight=None
)

ValueError: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: 103 (of type <class 'int'>)

In [21]:
#lstm only - slow
from tensorflow.keras.layers import Input
from tensorflow.keras import Model

def lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    
    x = Bidirectional(
        LSTM(256, return_sequences=True, 
                          kernel_regularizer=l2(0.001),
                          recurrent_regularizer=l2(0.001)))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    x = Bidirectional(LSTM(128, return_sequences=False,
                          kernel_regularizer=l2(0.001),
                          recurrent_regularizer=l2(0.001)))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.15)(x)
    
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.1)(x)
    
    outputs = Dense(1, kernel_regularizer=l2(0.001))(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
        loss='mae',
        metrics=['mae', 'mse']
    )
    
    return model

m = lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))

callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, min_delta=0.001),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, min_delta=0.001)
]

history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=128,
    callbacks=callbacks,
    class_weight=None
)

Epoch 1/150
  30/1611 ━━━━━━━━━━━━━━━━━━━━ 1:25:38 3s/step - loss: 6.5439 - mae: 4.5236 - mse: 34.7313

KeyboardInterrupt: 

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(
    onehot_1_np, log_fc_1_np, test_size=0.20, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Mean: {np.mean(y_train):.4f}")
print(f"Std: {np.std(y_train):.4f}")
print(f"Min: {np.min(y_train):.4f}")
print(f"Max: {np.max(y_train):.4f}")
print(f"Median: {np.median(y_train):.4f}")
print(f"IQR: {np.percentile(y_train, 75) - np.percentile(y_train, 25):.4f}")

Mean: -0.7595
Std: 5.7405
Min: -34.4414
Max: 36.1173
Median: -0.2336
IQR: 7.1796


In [15]:
#double model
def model():
    inputs = Input(shape=(203, 4))
    
    x = Conv1D(128, kernel_size=6, activation='relu', padding='same',
               kernel_regularizer=l2(0.001))(inputs)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)
    
    x = Conv1D(256, kernel_size=12, activation='relu', padding='same',
               kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.2)(x)
    
    x = Bidirectional(
        LSTM(128, return_sequences=False,
                          kernel_regularizer=l2(0.001)))(x)
    x = Dropout(0.2)(x)
    
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.15)(x)
    
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.1)(x)
    
    outputs = Dense(1, kernel_regularizer=l2(0.001))(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=Adam(learning_rate=1e-3, clipnorm=1.0),
        loss='huber',
        metrics=['mae', 'mse']
    )
    
    return model

callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, min_delta=0.001),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, min_delta=0.001)
]
m = model()

history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128, 
    callbacks=callbacks,
    class_weight=None
)

Epoch 1/50
 179/1549 ━━━━━━━━━━━━━━━━━━━━ 37:29 2s/step - loss: 4.5014 - mae: 4.4481 - mse: 32.9145

KeyboardInterrupt: 

In [ ]:
#model idek scaled

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).reshape(-1)
y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).reshape(-1)
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).reshape(-1)

def deep_model():
    inputs = Input(shape=(103, 4))
    
    conv1 = Conv1D(128, 3, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(inputs)
    conv1 = BatchNormalization()(conv1)
    conv1 = MaxPooling1D(2)(conv1)
    
    conv2 = Conv1D(128, 7, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(inputs)
    conv2 = BatchNormalization()(conv2)
    conv2 = MaxPooling1D(2)(conv2)
    
    conv3 = Conv1D(128, 11, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(inputs)
    conv3 = BatchNormalization()(conv3)
    conv3 = MaxPooling1D(2)(conv3)
    
    merged = Concatenate()([conv1, conv2, conv3])
    
    x = Conv1D(256, 9, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(merged)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Conv1D(256, 9, activation='relu', padding='same', kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(0.0001)))(x)
    x = Dropout(0.3)(x)
    
    x = Bidirectional(LSTM(64, return_sequences=False, kernel_regularizer=l2(0.0001)))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = Dropout(0.2)(x)
    
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = Dropout(0.1)(x)
    
    outputs = Dense(1, kernel_regularizer=l2(0.0001))(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=1e-3, clipnorm=1.0), loss='huber', metrics=['mae'])
    return model

model = deep_model()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-7)
]

print(f"Mean: {np.mean(y_train_scaled):.4f}")
print(f"Std: {np.std(y_train_scaled):.4f}")
print(f"Min: {np.min(y_train_scaled):.4f}")
print(f"Max: {np.max(y_train_scaled):.4f}")
print(f"Median: {np.median(y_train_scaled):.4f}")
print(f"IQR: {np.percentile(y_train_scaled, 75) - np.percentile(y_train_scaled, 25):.4f}")

history = model.fit(
    X_train, y_train_scaled,
    validation_data=(X_val, y_val_scaled),
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

In [17]:
def model(input_shape):
    inputs = Input(shape=input_shape)
    
    x = Conv1D(64, kernel_size=2, activation='relu', padding='same')(inputs)
    x = Dropout(0.1)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = Dropout(0.1)(x)

    x = LSTM(128, return_sequences=False,recurrent_dropout=0.25)(x)
    x = Dropout(0.1)(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-6))(x)
    x = Dropout(0.05)(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(1e-6))(x)

    outputs = Dense(1)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae', 'mse']
    )

    return model

m = model(input_shape=(X_train.shape[1], X_train.shape[2]))

callbacks = [
    EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=12, min_lr=1e-7)
]

m.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 203, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 203, 64)        │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 203, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 203, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 203, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 167,233 (653.25 KB)

 Trainable params: 167,233 (653.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=callbacks,
    class_weight=None
)

Epoch 1/20
  88/1549 ━━━━━━━━━━━━━━━━━━━━ 23:08 950ms/step - loss: 3.9746 - mae: 4.4504 - mse: 33.2262

In [ ]:
#eval
y_pred_val_scaled = model.predict(X_val, batch_size=512)
y_pred_val = y_scaler.inverse_transform(y_pred_val_scaled).reshape(-1)

y_pred_test_scaled = model.predict(X_test, batch_size=512)
y_pred_test = y_scaler.inverse_transform(y_pred_test_scaled).reshape(-1)

mae_val = mean_absolute_error(y_val, y_pred_val)
r2_val = r2_score(y_val, y_pred_val)
pearson_val = pearsonr(y_val, y_pred_val)[0]

print(f"Validation MAE: {mae_val:.3f}")
print(f"Validation R²: {r2_val:.3f}")
print(f"Validation Pearson: {pearson_val:.3f}")

within_2 = (np.abs(y_val - y_pred_val) <= 2.0).mean() * 100
within_3 = (np.abs(y_val - y_pred_val) <= 3.0).mean() * 100
print(f"Within ±2 FC: {within_2:.1f}%")
print(f"Within ±3 FC: {within_3:.1f}%")

mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)
pearson_test = pearsonr(y_test, y_pred_test)[0]

print(f"\nTest MAE: {mae_test:.3f}")
print(f"Test R²: {r2_test:.3f}")
print(f"Test Pearson: {pearson_test:.3f}")

In [ ]:
#4.42 mae
def model(input_shape):
    inputs = Input(shape=input_shape)
    
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.1)(x)

    x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
    x = Dropout(0.1)(x)

    x = Bidirectional(LSTM(128, return_sequences=False,recurrent_dropout=0.25))(x)
    x = Dropout(0.1)(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = Dropout(0.05)(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(1e-5))(x)

    outputs = Dense(1)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae', 'mse']
    )

    return model

m = model(input_shape=(X_train.shape[1], X_train.shape[2]))

history = m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=callbacks,
    class_weight=None
)

Epoch 1/20
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 609s 374ms/step - loss: 3.9732 - mae: 4.4484 - mse: 33.1093 - val_loss: 3.9606 - val_mae: 4.4365 - val_mse: 32.7475 - learning_rate: 0.0010
Epoch 2/20
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 620s 372ms/step - loss: 3.9700 - mae: 4.4455 - mse: 33.0397 - val_loss: 3.9565 - val_mae: 4.4322 - val_mse: 32.6229 - learning_rate: 0.0010
Epoch 3/20
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 615s 368ms/step - loss: 3.9681 - mae: 4.4438 - mse: 33.0305 - val_loss: 3.9576 - val_mae: 4.4339 - val_mse: 32.4951 - learning_rate: 0.0010
Epoch 4/20
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 628s 372ms/step - loss: 3.9668 - mae: 4.4428 - mse: 33.0044 - val_loss: 3.9573 - val_mae: 4.4337 - val_mse: 32.4765 - learning_rate: 0.0010
Epoch 5/20
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 630s 377ms/step - loss: 3.9655 - mae: 4.4415 - mse: 33.0001 - val_loss: 3.9574 - val_mae: 4.4338 - val_mse: 32.7875 - learning_rate: 0.0010
Epoch 6/20
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 620s 376ms/step - loss: 3.9627 - mae: 4.4387 - m

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1300/1611 ━━━━━━━━━━━━━━━━━━━━ 1:54 369ms/step - loss: 3.9495 - mae: 4.4249 - mse: 32.8556

In [17]:
#grid search
def model(lstm_units, dropout_rate1, dropout_rate2, dense_units, learning_rate, input_shape):
    model = Sequential([
        LSTM(lstm_units, input_shape=input_shape, return_sequences=False),
        Dropout(dropout_rate1),
        Dense(dense_units, activation='relu'),
        Dropout(dropout_rate2),
        Dense(1)
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )
    
    return model


def grid_search(X_train, y_train, X_val, y_val, param_grid, epochs=100, batch_size=32):
    keys = param_grid.keys()
    values = param_grid.values()
    param_combinations = [dict(zip(keys, v)) for v in product(*values)]
    
    results = []
    best_score = float('inf')
    best_params = None
    best_model = None
    
    for i, params in enumerate(param_combinations):
        m = model(
            lstm_units=params['lstm_units'],
            dropout_rate1=params['dropout_rate1'],
            dropout_rate2=params['dropout_rate2'],
            dense_units=params['dense_units'],
            learning_rate=params['learning_rate'],
            input_shape=(X_train.shape[1], X_train.shape[2])
        )
        
        callbacks = [
            EarlyStopping(
                monitor='val_loss', 
                patience=15, 
                restore_best_weights=True,
                verbose=0
            ),
            ReduceLROnPlateau(
                monitor='val_loss', 
                factor=0.5, 
                patience=7, 
                min_lr=1e-6,
                verbose=0
            )
        ]
        
        history = m.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=0
        )
        
        best_val_loss = min(history.history['val_loss'])
        best_val_mae = min(history.history['val_mae'])
        
        results.append({
            'params': params,
            'val_loss': best_val_loss,
            'val_mae': best_val_mae,
            'epochs_trained': len(history.history['loss'])
        })
        
        if best_val_loss < best_score:
            best_score = best_val_loss
            best_params = params
            best_model = m
    
    return best_params, best_model, results


param_grid = {
    'lstm_units': [64, 128, 256],
    'dropout_rate1': [0.2, 0.3, 0.4],
    'dropout_rate2': [0.1, 0.2, 0.3],
    'dense_units': [32, 64, 128],
    'learning_rate': [1e-4, 1e-3, 1e-2]
}

best_params, best_model, results = grid_search(
    X_train, y_train,
    X_val, y_val,
    param_grid=param_grid,
    epochs=100,
    batch_size=32
)

/cephfs/volumes/hpc_home/k20029203/fca66952-06d6-4b07-b95c-4b56cba53523/jvenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


KeyboardInterrupt: 